In [2]:
# Apple Silicon (M1/M2+) GPU support via Metal
# If you installed `tensorflow-macos` and `tensorflow-metal`, this will use GPU:0 automatically.
# Tip: conda/mamba env (example):
#   conda create -n tf-mps python=3.10 -y
#   conda activate tf-mps
#   pip install tensorflow-macos tensorflow-metal
import os, platform, sys
import tensorflow as tf

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("TensorFlow:", tf.__version__)

# Show devices
gpus = tf.config.list_physical_devices('GPU')
cpus = tf.config.list_physical_devices('CPU')
print("CPUs:", cpus)
print("GPUs:", gpus)

# Enable memory growth on detected GPUs (Metal backend shows up as GPU:0)
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print("Could not set memory growth:", e)

if gpus:
    print("Using GPU acceleration (Metal).")
else:
    print("No GPU detected. On Apple Silicon, install `tensorflow-macos` and `tensorflow-metal`.")


Python: 3.10.18
Platform: macOS-15.6.1-arm64-arm-64bit
TensorFlow: 2.16.2
CPUs: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Using GPU acceleration (Metal).


In [3]:
# Optional: Enable mixed precision (can speed up training on Apple GPUs)
# Note: If you see numerical instability, comment this out.
try:
    from tensorflow.keras import mixed_precision
    if tf.config.list_physical_devices('GPU'):
        mixed_precision.set_global_policy('mixed_float16')
        print("Mixed precision policy set to 'mixed_float16'.")
    else:
        print("Mixed precision not enabled (no GPU detected).")
except Exception as e:
    print("Could not enable mixed precision:", e)


Mixed precision policy set to 'mixed_float16'.


# BLSTM Translator (Amharic/Ge'ez/English) — Colab Notebook
Robust, trainable seq2seq with **Bidirectional LSTM encoder + attention decoder**. Includes data loading, training, checkpointing, evaluation (BLEU), and batch inference. Works with CSV columns like `amh, gez, eng`.

**Tip:** Runtime → Change runtime type → GPU.

In [4]:
import tensorflow as tf
#@title 2. Reinstall NumPy → TensorFlow → pandas → sacrebleu
# 1. NumPy first (exact version you asked for)
# (Tip) Consider installing these packages in your environment beforehand.

# 2. TensorFlow (will see the NumPy we just installed)
# (Tip) Consider installing these packages in your environment beforehand.

# 3. pandas & sacrebleu
# (Tip) Consider installing these packages in your environment beforehand.

# 4. Quick sanity-check
import numpy as np, pandas as pd, tensorflow as tf, sacrebleu
print(f"NumPy:     {np.__version__}")
print(f"Pandas:    {pd.__version__}")
print(f"TensorFlow:{tf.__version__}")
print(f"SacreBLEU: {sacrebleu.__version__}")
print("All imports succeeded!")
print("\nDevice check:", tf.config.list_physical_devices("GPU"))
if not tf.config.list_physical_devices("GPU"):
    print(" No GPU visible to TensorFlow. On Apple Silicon, install `tensorflow-macos` and `tensorflow-metal`.")


NumPy:     1.26.4
Pandas:    2.2.2
TensorFlow:2.16.2
SacreBLEU: 2.5.1
All imports succeeded!

Device check: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
#@title Setup (installs & imports)
AUTO_INSTALL_PACKAGES = False  #@param {type:"boolean"}

if AUTO_INSTALL_PACKAGES:
    !python -m pip install --upgrade pip
    !pip -q install tensorflow-macos==2.15.1 tensorflow-metal==1.1.0 sacrebleu==2.4.0 pandas==2.2.2 numpy==1.26.4 --no-warn-script-location

import os, math, random, json, pickle, gc
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dense, AdditiveAttention, Concatenate, TimeDistributed, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras import mixed_precision

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    files = None
    IN_COLAB = False

import sacrebleu

if IN_COLAB:
    print("Colab detected: you can use files.upload().")
else:
    print("Running outside Colab; make sure local data paths exist.")


Running outside Colab; make sure local data paths exist.


In [6]:
print('TensorFlow:', tf.__version__)
try:
    from tensorflow.python.client import device_lib
    print('Devices:', [d.name for d in device_lib.list_local_devices()])
except Exception as e:
    print('Device check skipped:', e)

# Optional: enable mixed precision for speed on Apple GPUs
USE_MIXED_PRECISION = False  #@param {type:"boolean"}
if USE_MIXED_PRECISION:
    policy = mixed_precision.Policy('mixed_float16')
    mixed_precision.set_global_policy(policy)
    print("Mixed precision:", mixed_precision.global_policy())
else:
    print("Mixed precision: disabled")

# Optional: gradient clipping
CLIP_NORM = 1.0  #@param {type:"number"}


TensorFlow: 2.16.2
Devices: ['/device:CPU:0', '/device:GPU:0']
Mixed precision: disabled


2025-11-08 20:48:51.562294: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2025-11-08 20:48:51.562413: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2025-11-08 20:48:51.562786: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2025-11-08 20:48:51.563354: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-11-08 20:48:51.563855: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


## Upload your CSV
Upload a CSV with columns like `amh, gez, eng`. You can train any direction by choosing `src_col` and `tgt_col` below.

In [7]:
#@title Data source (local CSV path)
from pathlib import Path

csv_path = "data/AGE.csv"  #@param {type:"string"}
csv_path = Path(csv_path).expanduser()

if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found at {csv_path}. Update `csv_path` to point to your local file.")

print("Using:", csv_path)


Using: data/AGE.csv


## Setting the Hyper-Parameters
Tune training lengths, epochs, and vocab sizes to fit your GPU/time.

In [8]:
#@title Config
src_col = "gez"  #@param {type:"string"}
tgt_col = "amh"  #@param {type:"string"}
sample_frac = None  #@param {type:"number"} # set to None to use the full dataset
epochs = 20  #@param {type:"integer"}
batch_size = 128  #@param {type:"integer"}
max_src_len = 60  #@param {type:"integer"}
max_tgt_len = 60  #@param {type:"integer"}
src_vocab = 20000  #@param {type:"integer"} # set 0 to auto-use all tokens
tgt_vocab = 20000  #@param {type:"integer"} # set 0 to auto-use all tokens
emb_dim = 250  #@param {type:"integer"}
enc_units = 250  #@param {type:"integer"}
dec_units = 250  #@param {type:"integer"}
learning_rate = 0.001  #@param {type:"number"}
use_adaptive_lr = True  #@param {type:"boolean"}
adaptive_lr_factor = 0.5  #@param {type:"number"}
adaptive_lr_patience = 1  #@param {type:"integer"}
adaptive_lr_min = 5e-5  #@param {type:"number"}
max_train_batches = 0  #@param {type:"integer"} # <=0 means auto-fill all batches per epoch
max_val_batches = 0  #@param {type:"integer"} # <=0 means auto-fill all batches per epoch
val_split = 0.1  #@param {type:"number"}
dropout = 0.2  #@param {type:"number"}

auto_vocab = True  #@param {type:"boolean"} # override caps with full vocab when True

out_dir = "/content/drive/MyDrive/geez-amharic-eng-translator/artifacts"
os.makedirs(out_dir, exist_ok=True)

def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

seed_all(42)


## Load & Preview Data

In [9]:

#@title Load & peek
def read_data(csv_path, src_col, tgt_col, sample_frac=None):
    df = pd.read_csv(csv_path)
    if src_col not in df.columns or tgt_col not in df.columns:
        raise ValueError(f"CSV must contain columns '{src_col}' and '{tgt_col}'. Found: {list(df.columns)}")
    df = df[[src_col, tgt_col]].dropna()
    if sample_frac is not None and 0 < sample_frac < 1.0:
        df = df.sample(frac=sample_frac, random_state=42)
    df[src_col] = df[src_col].astype(str).str.strip()
    df[tgt_col] = df[tgt_col].astype(str).str.strip()
    return df.reset_index(drop=True)

df = read_data(csv_path, src_col, tgt_col, sample_frac=sample_frac)
print(f"Pairs: {len(df)}")
display(df.head(5))


Pairs: 1750


,gez,amh
0,ወእመሰ አልቦ ውስተ እዴሁ ወኢየአክል ለዘውገ መዓንቅ ወለክልኤቱ እጕለ ር...,ሁለት ዋኖሶች ወይም ሁለት የርግብ ግልገሎች ለማምጣት ገንዘቡ ያልበቃ እን...
1,ወባሕቱ ከመ ኢያንጐርጕሩ ሑር ውስተ ባሕር ወደይ መቃጥነ ወዘቀዳሚ አሥገር...,ነገር ግን እንዳናሰናክላቸው፥ ወደ ባሕር ሂድና መቃጥን ጣል፤ መጀመሪያም ...
2,ወአበሴሎምሰ ተኀጥአ ወእምዝ አልዐለ አዕይንቲሁ ወልድ ዘይኔጽር ወሶበ ይሬ...,አቤሴሎምም ኰበለለ። ጕበኛውም ጕልማሳ ዓይኑን ከፍ አደረገ፥ እነሆም፥ ብዙ...
3,ወእንዘ ያስተዋድይዎ ሊቃነ ካህናት ወሊቃውንተ ሕዝብ አልቦ ዘተሰጥዎሙ ወኢ...,የካህናት አለቆችም ሽማግሎችም ሲከሱት ምንም አልመለሰም።
4,ወአውሥኦሙ በለዓም ወይቤሎሙ ለመላእክተ በላቅ ምልአ ቤት ወርቀ ወብሩረ እ...,በለዓምም ሲነጋ ተነሣ፥ አህያይቱንም ጭኖ ከሞዓብ አለቆች ጋር ሄደ።


## Tokenization
Whitespace tokenization that preserves Ethiopic scripts. We add `<s>` and `</s>` around the target for teacher forcing.

In [10]:
#@title Tokenizers
def prepare_tokenizers(src_texts, tgt_texts, num_words_src=None, num_words_tgt=None, oov_token="<unk>"):
    num_words_src = None if (num_words_src is None or num_words_src <= 0) else num_words_src
    num_words_tgt = None if (num_words_tgt is None or num_words_tgt <= 0) else num_words_tgt
    src_tok = Tokenizer(num_words=num_words_src, filters="", lower=False, oov_token=oov_token, split=" ")
    tgt_tok = Tokenizer(num_words=num_words_tgt, filters="", lower=False, oov_token=oov_token, split=" ")
    tgt_in_texts  = [f"<s> {t}" for t in tgt_texts]
    tgt_out_texts = [f"{t} </s>" for t in tgt_texts]
    src_tok.fit_on_texts(src_texts)
    tgt_tok.fit_on_texts(tgt_in_texts + tgt_out_texts)
    return src_tok, tgt_tok, tgt_in_texts, tgt_out_texts

src_tok, tgt_tok, tgt_in_texts, tgt_out_texts = prepare_tokenizers(
    df[src_col].tolist(), df[tgt_col].tolist(), src_vocab, tgt_vocab
)

src_vocab_full = len(src_tok.word_index) + 1
tgt_vocab_full = len(tgt_tok.word_index) + 1

def _effective_vocab(user_cap, full_vocab):
    if auto_vocab or user_cap <= 0:
        return full_vocab
    return min(user_cap, full_vocab)

src_vocab_effective = _effective_vocab(src_vocab, src_vocab_full)
tgt_vocab_effective = _effective_vocab(tgt_vocab, tgt_vocab_full)

# dynamic lengths (bounded by user caps)
dyn_max_src = min(max_src_len, max((len(s.split()) for s in df[src_col]), default=1))
dyn_max_tgt = min(max_tgt_len, max((len(s.split())+2 for s in df[tgt_col]), default=1))
dyn_max_src = max(dyn_max_src, 4)
dyn_max_tgt = max(dyn_max_tgt, 4)

def texts_to_padded(tokenizer, texts, maxlen):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=maxlen, padding="post", truncating="post")

X_src = texts_to_padded(src_tok, df[src_col].tolist(), dyn_max_src)
X_tgt_in = texts_to_padded(tgt_tok, tgt_in_texts, dyn_max_tgt)
X_tgt_out = texts_to_padded(tgt_tok, tgt_out_texts, dyn_max_tgt)
y = np.expand_dims(X_tgt_out, -1)

print("dyn_max_src:", dyn_max_src, "dyn_max_tgt:", dyn_max_tgt)
print("src vocab size (effective):", src_vocab_effective, "/ full:", src_vocab_full)
print("tgt vocab size (effective):", tgt_vocab_effective, "/ full:", tgt_vocab_full)


dyn_max_src: 45 dyn_max_tgt: 50
src vocab size: 10000
tgt vocab size: 10000


In [ ]:
#@title Debug: vocab + length stats
src_seen = len(src_tok.word_index)
tgt_seen = len(tgt_tok.word_index)
print(f"Source vocab seen: {src_seen} (cap={src_vocab} | effective={src_vocab_effective})")
print(f"Target vocab seen: {tgt_seen} (cap={tgt_vocab} | effective={tgt_vocab_effective})")
print(f"dyn_max_src: {dyn_max_src} | dyn_max_tgt: {dyn_max_tgt}")
print(f"Samples: {len(df)} | Validation split: {val_split}")

if src_vocab > 0 and not auto_vocab and src_seen + 1 > src_vocab:
    coverage = src_vocab / (src_seen + 1)
    print(f"[Warn] Source vocab capped: covers {coverage:.2%} of tokens.")
if tgt_vocab > 0 and not auto_vocab and tgt_seen + 1 > tgt_vocab:
    coverage = tgt_vocab / (tgt_seen + 1)
    print(f"[Warn] Target vocab capped: covers {coverage:.2%} of tokens.")

from collections import Counter
src_lengths = Counter((len(s.split()) for s in df[src_col]))
tgt_lengths = Counter((len(s.split()) for s in df[tgt_col]))
print(f"Most common src lengths: {src_lengths.most_common(5)}")
print(f"Most common tgt lengths: {tgt_lengths.most_common(5)}")


In [11]:
#@title Build tf.data pipelines (optional)
USE_TF_DATA = True  #@param {type:"boolean"}
TF_DATA_BUFFER = 4096  #@param {type:"integer"}

train_dataset = None
val_dataset = None

train_cap = None if max_train_batches is None or max_train_batches <= 0 else int(max_train_batches)
val_cap = None if max_val_batches is None or max_val_batches <= 0 else int(max_val_batches)

if USE_TF_DATA:
    data_size = len(X_src)
    if data_size == 0:
        raise ValueError("No samples to build datasets.")
    if not 0 < val_split < 1:
        raise ValueError("val_split must be between 0 and 1 when USE_TF_DATA is True.")
    val_count = max(1, int(data_size * val_split))
    train_count = data_size - val_count
    if train_count <= 0:
        raise ValueError("val_split too high for current dataset size.")
    rng = np.random.default_rng(42)
    idx = rng.permutation(data_size)
    train_idx = idx[:train_count]
    val_idx = idx[train_count:]

    def make_dataset(indices, training=True):
        ds = tf.data.Dataset.from_tensor_slices(((X_src[indices], X_tgt_in[indices]), y[indices]))
        if training:
            ds = ds.shuffle(min(TF_DATA_BUFFER, len(indices)), seed=42, reshuffle_each_iteration=True)
        ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
        return ds

    train_dataset = make_dataset(train_idx, training=True)
    val_dataset = make_dataset(val_idx, training=False)

    if train_cap is not None:
        train_dataset = train_dataset.take(max(1, train_cap))
    if val_cap is not None:
        val_dataset = val_dataset.take(max(1, val_cap))

if train_dataset is None:
    print("Using NumPy arrays directly (validation_split handled in model.fit).")
else:
    def _fmt_batches(ds):
        card = tf.data.experimental.cardinality(ds).numpy()
        return 'unknown' if card < 0 else int(card)
    print(f"tf.data pipelines ready → train batches: {_fmt_batches(train_dataset)}, val batches: {_fmt_batches(val_dataset)}")


tf.data pipelines ready → train batches: 197, val batches: 22


2025-11-08 20:49:06.044789: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-11-08 20:49:06.044835: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


## Model: BLSTM Encoder + Attention Decoder

In [14]:
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras import mixed_precision
import tensorflow as tf

mixed_precision.set_global_policy('mixed_float16')  # if you're on Metal

def _apply_src_mask_fp32(scores, src_mask):
    s32 = tf.cast(scores, tf.float32)
    m32 = tf.cast(src_mask, tf.float32)
    m32 = tf.expand_dims(m32, axis=1)
    s32 = s32 + (1.0 - m32) * (-1e4)          # avoid -inf in fp16
    return s32

def _softmax_fp32(logits, axis=-1):
    return tf.nn.softmax(tf.cast(logits, tf.float32), axis=axis)

class NoMask(Layer):
    def call(self, x):
        return x
    def compute_mask(self, inputs, mask=None):
        # Explicitly drop mask so nothing downstream receives it
        return None


def build_model(
    src_vocab_size, tgt_vocab_size,
    emb_dim=192, enc_units=192, dec_units=192,
    max_src_len=80, max_tgt_len=80, dropout=0.2, clip_norm=1.0, learning_rate=3e-4
):
    src_in = Input(shape=(max_src_len,), name="src_in")
    tgt_in = Input(shape=(max_tgt_len,), name="tgt_in")

    enc_emb = Embedding(src_vocab_size, emb_dim, mask_zero=False, name="src_emb")(src_in)
    dec_emb = Embedding(tgt_vocab_size, emb_dim, mask_zero=False, name="tgt_emb")(tgt_in)

    enc_blstm, f_h, f_c, b_h, b_c = Bidirectional(
        LSTM(enc_units, return_sequences=True, return_state=True,
             implementation=2, dropout=0.0, recurrent_dropout=0.0, name="enc_lstm"),
        name="bilstm"
    )(enc_emb)

    state_h = Concatenate(name="enc_state_h")([f_h, b_h])
    state_c = Concatenate(name="enc_state_c")([f_c, b_c])
    state_h = Dense(dec_units, activation="tanh", name="map_h")(state_h)
    state_c = Dense(dec_units, activation="tanh", name="map_c")(state_c)

    dec_out, _, _ = LSTM(dec_units, return_sequences=True, return_state=True,
                         implementation=2, dropout=0.0, recurrent_dropout=0.0,
                         name="dec_lstm")(dec_emb, initial_state=[state_h, state_c])

    # Bahdanau-style but fp32-safe path
    Wq = TimeDistributed(Dense(dec_units, use_bias=False), name="Wq")(dec_out)      # (B, Tq, D)
    Wk = TimeDistributed(Dense(dec_units, use_bias=False), name="Wk")(enc_blstm)    # (B, Tk, D)

    Wq_e = Lambda(lambda x: tf.expand_dims(x, axis=2), name="expand_Wq")(Wq)
    Wk_e = Lambda(lambda x: tf.expand_dims(x, axis=1), name="expand_Wk")(Wk)
    e_tanh = Lambda(lambda x: tf.tanh(x[0] + x[1]), name="e_tanh")([Wq_e, Wk_e])    # (B,Tq,Tk,D)

    e = TimeDistributed(TimeDistributed(Dense(1, use_bias=True)), name="score")(e_tanh)
    e = Lambda(lambda x: tf.squeeze(x, axis=-1), name="score_squeeze")(e)           # (B,Tq,Tk)

    tgt_mask = Lambda(lambda x: tf.not_equal(x, 0), name="tgt_mask")(tgt_in)
    src_mask = Lambda(lambda x: tf.not_equal(x, 0), name="src_mask")(src_in)

    e_masked_fp32 = Lambda(lambda xs: _apply_src_mask_fp32(xs[0], xs[1]),
                           name="apply_src_mask")([e, src_mask])

    alphas = Lambda(lambda x: _softmax_fp32(x, axis=-1), name="alphas")(e_masked_fp32)  # (B,Tq,Tk)

    context = Lambda(lambda x: tf.matmul(x[0], x[1]), name="context")([alphas, enc_blstm])  # (B,Tq,2*enc_units)

    def _mask_context(args):
        ctx, qmask = args
        qmask = tf.cast(qmask, ctx.dtype)
        return ctx * tf.expand_dims(qmask, axis=-1)

    context = Lambda(_mask_context, name="mask_context")([context, tgt_mask])

    comb = Concatenate(name="attn_concat")([dec_out, context])
    comb = Dropout(dropout, name="dropout")(comb)
    comb = NoMask(name="drop_mask")(comb)

    logits = TimeDistributed(Dense(tgt_vocab_size, dtype="float32"), name="logits")(comb)

    model = Model([src_in, tgt_in], logits)

    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    opt = tf.keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=clip_norm or None)
    model.compile(optimizer=opt, loss=loss_fn, metrics=[])
    return model


In [15]:
model = build_model(
    src_vocab_size=src_vocab_effective,
    tgt_vocab_size=tgt_vocab_effective,
    emb_dim=emb_dim, enc_units=enc_units, dec_units=dec_units,
    max_src_len=dyn_max_src, max_tgt_len=dyn_max_tgt,
    dropout=dropout, clip_norm=CLIP_NORM, learning_rate=learning_rate
)
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ src_in (InputLayer) │ (None, 45)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ src_emb (Embedding) │ (None, 45, 128)   │  1,280,000 │ src_in[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm              │ [(None, 45, 256), │    263,168 │ src_emb[0][0]     │
│ (Bidirectional)     │ (None, 128),      │            │                   │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tgt_in (InputLayer) │ (None, 50)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_state_h         │ (None, 256)       │          0 │ bilstm[0][1],     │
│ (Concatenate)       │                   │            │ bilstm[0][3]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_state_c         │ (None, 256)       │          0 │ bilstm[0][2],     │
│ (Concatenate)       │                   │            │ bilstm[0][4]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tgt_emb (Embedding) │ (None, 50, 128)   │  1,280,000 │ tgt_in[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ map_h (Dense)       │ (None, 128)       │     32,896 │ enc_state_h[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ map_c (Dense)       │ (None, 128)       │     32,896 │ enc_state_c[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_lstm (LSTM)     │ [(None, 50, 128), │    131,584 │ tgt_emb[0][0],    │
│                     │ (None, 128),      │            │ map_h[0][0],      │
│                     │ (None, 128)]      │            │ map_c[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Wq                  │ (None, 50, 128)   │     16,384 │ dec_lstm[0][0]    │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Wk                  │ (None, 45, 128)   │     32,768 │ bilstm[0][0]      │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_Wq (Lambda)  │ (None, 50, 1,     │          0 │ Wq[0][0]          │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_Wk (Lambda)  │ (None, 1, 45,     │          0 │ Wk[0][0]          │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ e_tanh (Lambda)     │ (None, 50, 45,    │          0 │ expand_Wq[0][0],  │
│                     │ 128)              │            │ expand_Wk[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ score               │ (None, 50, 45, 1) │        129 │ e_tanh[0][0]      │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ score_squeeze       │ (None, 50, 45)    │          0 │ score[0][0]       │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 6,919,825 (26.40 MB)

 Trainable params: 6,919,825 (26.40 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# ensure integer type and proper shape
y = np.expand_dims(X_tgt_out, -1).astype("int32")  # or int64


In [17]:
opt = tf.keras.optimizers.Adam(learning_rate=3e-4, clipnorm=CLIP_NORM)
# or lower LR slightly


## Train
Early stopping + model checkpointing (best weights).

In [ ]:
#@title Debug: gradient norms (small subset)
DEBUG_BATCHES = 8  #@param {type:"integer"}
MAX_NAN_WARNINGS = 3  #@param {type:"integer"}

import matplotlib.pyplot as plt

batch_iter = tf.data.Dataset.from_tensor_slices(((X_src, X_tgt_in), y))    .batch(batch_size)    .take(max(1, DEBUG_BATCHES))

loss_obj = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
step_losses, grad_norms = [], []
nan_hits = 0

for step, batch in enumerate(batch_iter):
    (src_batch, tgt_batch), y_batch = batch
    with tf.GradientTape() as tape:
        logits = model([src_batch, tgt_batch], training=True)
        loss_value = loss_obj(y_batch, logits)
    grads = [g for g in tape.gradient(loss_value, model.trainable_variables) if g is not None]
    grad_norm = tf.linalg.global_norm(grads)
    loss_float = float(loss_value.numpy())
    grad_float = float(grad_norm.numpy()) if grad_norm is not None else float('nan')
    step_losses.append(loss_float)
    grad_norms.append(grad_float)
    if not (np.isfinite(loss_float) and np.isfinite(grad_float)):
        nan_hits += 1
        print(f"Warning: non-finite values at step {step}: loss={loss_float}, grad_norm={grad_float}")
        if nan_hits >= MAX_NAN_WARNINGS:
            break

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(step_losses, marker='o')
plt.title('Debug loss per batch')
plt.xlabel('Batch')
plt.ylabel('Loss')
plt.grid(True)

plt.subplot(1,2,2)
plt.plot(grad_norms, marker='x', color='orange')
plt.title('Gradient norm per batch')
plt.xlabel('Batch')
plt.ylabel('||grad||')
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"Collected {len(step_losses)} debug batches.")


In [ ]:
#@title Train
ckpt_path = os.path.join(out_dir, "best_model.keras")
callbacks=[
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True, save_weights_only=False)
]

if use_adaptive_lr:
    callbacks.append(
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=adaptive_lr_factor,
            patience=adaptive_lr_patience,
            min_lr=adaptive_lr_min,
            verbose=1
        )
    )

fit_kwargs = dict(epochs=epochs, callbacks=callbacks, verbose=2)

train_cap = None if max_train_batches is None or max_train_batches <= 0 else int(max_train_batches)
if train_cap is not None:
    train_examples = int(len(X_src) * (1.0 - val_split))
    batches_per_epoch = max(1, math.ceil(train_examples / batch_size))
    fit_kwargs["steps_per_epoch"] = min(train_cap, batches_per_epoch)

history = model.fit(
    [X_src, X_tgt_in], y,
    validation_split=val_split,
    batch_size=batch_size,
    **fit_kwargs
)
print("Best checkpoint saved to:", ckpt_path)


Epoch 1/8


2025-11-08 20:50:41.003385: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


## Save Tokenizers + Meta

In [ ]:

#@title Save artifacts
with open(os.path.join(out_dir, "src_tokenizer.pkl"), "wb") as f:
    pickle.dump(src_tok, f)
with open(os.path.join(out_dir, "tgt_tokenizer.pkl"), "wb") as f:
    pickle.dump(tgt_tok, f)
meta = {"src_col": src_col, "tgt_col": tgt_col, "max_src_len": int(dyn_max_src), "max_tgt_len": int(dyn_max_tgt)}
with open(os.path.join(out_dir, "meta.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print("Artifacts saved in:", out_dir)


## Greedy Decoder (Inference)

In [ ]:
#@title Greedy decode helpers
def _infer_src_shape(model):
    input_spec = model.input_shape
    if isinstance(input_spec, list):
        for spec in input_spec:
            if spec is not None:
                return spec[1:]
        raise ValueError("Cannot infer source shape from model.input_shape list.")
    if input_spec is None:
        raise ValueError("Model input shape is None; ensure the model is built.")
    return input_spec[1:]

def build_encoder_subgraph(model):
    src_shape = _infer_src_shape(model)
    src_input = tf.keras.Input(shape=src_shape, dtype="int32", name="enc_src_input")
    enc_emb = model.get_layer("src_emb")(src_input)
    bilstm = model.get_layer("bilstm")
    enc_blstm, f_h, f_c, b_h, b_c = bilstm(enc_emb)
    state_h = model.get_layer("enc_state_h")([f_h, b_h])
    state_c = model.get_layer("enc_state_c")([f_c, b_c])
    map_h = model.get_layer("map_h")(state_h)
    map_c = model.get_layer("map_c")(state_c)
    return tf.keras.Model(src_input, [enc_blstm, map_h, map_c], name="encoder_subgraph")

def greedy_decode(model, src_seq, src_tok, tgt_tok, max_src_len, max_tgt_len, enc_model=None):
    if len(src_seq) == 0:
        src_seq = [src_tok.word_index.get("<unk>", 0)]
    if enc_model is None:
        enc_model = build_encoder_subgraph(model)
    src_seq = pad_sequences([src_seq], maxlen=max_src_len, padding="post", dtype="int32")
    enc_outputs, h, c = enc_model.predict(src_seq, verbose=0)
    enc_outputs = tf.convert_to_tensor(enc_outputs)
    keys = model.get_layer("Wk")(enc_outputs)
    src_mask = model.get_layer("src_mask")(tf.convert_to_tensor(src_seq))

    dec_emb_layer = model.get_layer("tgt_emb")
    dec_lstm = model.get_layer("dec_lstm")
    Wq_layer = model.get_layer("Wq")
    expand_Wq = model.get_layer("expand_Wq")
    expand_Wk = model.get_layer("expand_Wk")
    e_tanh = model.get_layer("e_tanh")
    score_layer = model.get_layer("score")
    squeeze_layer = model.get_layer("score_squeeze")
    apply_mask = model.get_layer("apply_src_mask")
    softmax_layer = model.get_layer("alphas")
    context_layer = model.get_layer("context")
    attn_concat = model.get_layer("attn_concat")
    dropout = model.get_layer("dropout")
    logits_td = model.get_layer("logits")

    index2word = {i: w for w, i in tgt_tok.word_index.items()}
    index2word[0] = "<pad>"
    start_id = tgt_tok.word_index.get("<s>")
    end_id = tgt_tok.word_index.get("</s>")
    if start_id is None or end_id is None:
        raise RuntimeError("Missing <s> or </s> in target tokenizer.")

    y = np.array([[start_id]], dtype="int32")
    out_tokens = []
    for _ in range(max_tgt_len):
        y_emb = dec_emb_layer(y)
        dec_out, h, c = dec_lstm(y_emb, initial_state=[h, c])
        queries = Wq_layer(dec_out)
        Wq_e = expand_Wq(queries)
        Wk_e = expand_Wk(keys)
        e = e_tanh([Wq_e, Wk_e])
        scores = score_layer(e)
        scores = squeeze_layer(scores)
        masked_scores = apply_mask([scores, src_mask])
        alphas = softmax_layer(masked_scores)
        context = context_layer([alphas, enc_outputs])
        comb = attn_concat([dec_out, context])
        comb = dropout(comb, training=False)
        probs = logits_td(comb).numpy()
        next_id = int(np.argmax(probs[0, 0]))
        if next_id == end_id:
            break
        out_tokens.append(next_id)
        y = np.array([[next_id]], dtype="int32")
    out_words = [index2word.get(i, "<unk>") for i in out_tokens]
    return " ".join(out_words)

def translate_sentences(model, sentences, src_tok, tgt_tok, max_src_len, max_tgt_len):
    enc_model = build_encoder_subgraph(model)
    results = []
    for s in sentences:
        seqs = src_tok.texts_to_sequences([s])
        seq = seqs[0] if seqs and len(seqs[0]) else [src_tok.word_index.get("<unk>", 0)]
        pred = greedy_decode(model, seq, src_tok, tgt_tok, max_src_len, max_tgt_len, enc_model=enc_model)
        results.append(pred)
    return results


## Quick Sanity Translations

In [ ]:
#@title Teacher-forced diagnostics
TEACHER_FORCE_SAMPLES = 5  #@param {type:"integer"}
RANDOM_SEED = 7  #@param {type:"integer"}

index2word = {i: w for w, i in tgt_tok.word_index.items()}
index2word[0] = "<pad>"
start_id = tgt_tok.word_index.get("<s>")
end_id = tgt_tok.word_index.get("</s>")
if start_id is None or end_id is None:
    raise RuntimeError("Missing <s> or </s> in target tokenizer.")

rng = np.random.default_rng(RANDOM_SEED)
indices = rng.choice(len(df), size=min(TEACHER_FORCE_SAMPLES, len(df)), replace=False)

for i in indices:
    src_sentence = df.iloc[i][src_col]
    tgt_sentence = df.iloc[i][tgt_col]
    src_seq = src_tok.texts_to_sequences([src_sentence])
    src_seq = src_seq[0] if src_seq and len(src_seq[0]) else [src_tok.word_index.get("<unk>", 0)]
    src_seq = pad_sequences([src_seq], maxlen=dyn_max_src, padding="post", dtype="int32")

    tgt_input = tgt_tok.texts_to_sequences([f"<s> {tgt_sentence}"])[0]
    tgt_input = pad_sequences([tgt_input], maxlen=dyn_max_tgt, padding="post", dtype="int32")

    logits = model.predict([src_seq, tgt_input], verbose=0)
    pred_ids = logits[0].argmax(axis=-1)

    def _decode(ids):
        words = []
        for idx in ids:
            if idx == 0:
                continue
            if idx == end_id:
                break
            words.append(index2word.get(int(idx), "<unk>"))
        return " ".join(words)

    teacher_pred = _decode(pred_ids)

    print("SRC:", src_sentence)
    print("TGT:", tgt_sentence)
    print("Teacher-forced pred:", teacher_pred)
    print("-")

TF_METRIC_SAMPLES = 256  #@param {type:"integer"}
if TF_METRIC_SAMPLES > 0:
    subset = rng.choice(len(X_src), size=min(TF_METRIC_SAMPLES, len(X_src)), replace=False)
    logits = model.predict([X_src[subset], X_tgt_in[subset]], verbose=0)
    preds = logits.argmax(axis=-1)
    matches = (preds == X_tgt_out[subset])
    tf_acc = np.mean(matches)
    print(f"Teacher-forced token accuracy on subset: {tf_acc:.4f}")


In [ ]:

#@title Sample predictions
samples = df[src_col].tolist()[:5]
preds = translate_sentences(model, samples, src_tok, tgt_tok, dyn_max_src, dyn_max_tgt)
for s,p,t in zip(samples, preds, df[tgt_col].tolist()[:5]):
    print("SRC:", s)
    print("PRED:", p)
    print("TGT:", t)
    print("-"*80)


## BLEU Evaluation
On a small held-out split (random 200 by default).

In [ ]:

#@title Evaluate BLEU
eval_n = 200  #@param {type:"integer"}
idx = np.random.choice(len(df), size=min(eval_n, len(df)), replace=False)
refs = []
hyps = []
for i in idx:
    s = df.iloc[i][src_col]
    t = df.iloc[i][tgt_col]
    pred = translate_sentences(model, [s], src_tok, tgt_tok, dyn_max_src, dyn_max_tgt)[0]
    refs.append(t)
    hyps.append(pred)
# SacreBLEU expects list of hypothesis strings and list of reference-list(s)
bleu = sacrebleu.corpus_bleu(hyps, [refs])
print("BLEU:", bleu.score)


## Batch Translate a File
Upload a TXT with one source sentence per line.

In [ ]:
#@title Batch translate (TXT → CSV)
from datetime import datetime
from pathlib import Path

batch_txt_path = ""  #@param {type:"string"}
batch_txt_path = batch_txt_path.strip()

if batch_txt_path:
    txt_path = Path(batch_txt_path).expanduser()
    if not txt_path.exists():
        raise FileNotFoundError(f"TXT not found at {txt_path}. Update `batch_txt_path` or leave blank for Colab upload.")
else:
    if files is None:
        raise RuntimeError("Set `batch_txt_path` when running outside Colab.")
    print("Upload a .txt with one sentence per line...")
    uploaded_txt = files.upload()
    txt_path = Path(list(uploaded_txt.keys())[0])

with txt_path.open("r", encoding="utf-8") as f:
    src_lines = [line.strip() for line in f if line.strip()]

hyps = translate_sentences(model, src_lines, src_tok, tgt_tok, dyn_max_src, dyn_max_tgt)
out_csv = f"translations_{src_col}_to_{tgt_col}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
pd.DataFrame({src_col: src_lines, tgt_col: hyps}).to_csv(out_csv, index=False, encoding="utf-8")
print("Saved:", out_csv)


## Reload Saved Model
Load from checkpoint and reuse the same tokenizers.

In [ ]:

#@title Reload best checkpoint
reloaded = tf.keras.models.load_model(os.path.join(out_dir, "best_model.keras"))
print("Reloaded.")
# Sanity test
test_src = df[src_col].iloc[0]
print("SRC:", test_src)
print("PRED:", translate_sentences(reloaded, [test_src], src_tok, tgt_tok, dyn_max_src, dyn_max_tgt)[0])


## Tips for speed/robustness
- Use `sample_frac=0.3` while iterating.
- Reduce `epochs` to fit your time budget; add back later.
- Increase `batch_size` if you have more GPU memory.
- Mixed precision is enabled by default.
- Gradient clipping (`CLIP_NORM`) helps stabilize training.
- Use the checkpoint `.keras` file for the best model at inference time.